# Deputy Network Descriptive Results

Clean project version. Cell outputs were removed; datasets are read from `../data/` and generated files should go to `../outputs/`.



- [ 1)Importando os dados ](#secao1)


- [ 2)Tabelas de Correlação](#secao2)


- [ 3) Lista dos deputados com 100 maiores nós ](#secao3)


- [ 4)Lista dos deputados com maiores nós por orientação?](#secao4)

In [1]:
# Project paths
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
OUTPUT_DATA_DIR = OUTPUT_DIR / "data"

for _folder in (DATA_DIR, OUTPUT_DIR, FIGURE_DIR, TABLE_DIR, OUTPUT_DATA_DIR):
    _folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT.resolve()}")


Project root: /home/carlos/Carlos/Bahia_Clean_Project


In [2]:
# bilitoeca force atals e suas dependências
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx

<a id='secao1'></a>
## 1) Importando os dados 

### importando os dados da superplanilha

In [3]:
#plainha com as votações e os nomes dos deputados
superplanilha_com_cargos = pd.read_csv(DATA_DIR / 'superplanilha_com_cargos.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/home/carlos/Carlos/Bahia_Clean_Project/data/superplanilha_com_cargos.csv'

In [ ]:
#checking
superplanilha_com_cargos .head(1)

In [ ]:
presidentes_df = superplanilha_com_cargos.loc[superplanilha_com_cargos['cargo'] == 'Presidente']
presidentes_df.tail(50)


In [ ]:
# Supondo que presidentes_df é o seu DataFrame
# Filtrar as linhas onde o Nome é 'Arthur Lira' e a Legislatura é 57
arthur_lira_57_df = presidentes_df[(presidentes_df['Nome'] == 'Arthur Lira') & (presidentes_df['Legislatura'] == 57)]

# Contar o número de linhas que correspondem ao filtro
numero_de_linhas = arthur_lira_57_df.shape[0]

print("Número de linhas com 'Arthur Lira' e Legislatura '57':", numero_de_linhas)


In [ ]:
# Este código remove duplicatas baseado nas colunas 'Nome' e 'Legislatura'
# Mantém a primeira ocorrência de cada deputado por legislatura
presidentes_unicos_df = presidentes_df.drop_duplicates(subset=['Nome', 'Legislatura'], keep='first')

# Exibe as primeiras 50 linhas do novo DataFrame
presidentes_unicos_df.head(3)


In [ ]:
#ocupacoes = pd.read_csv('ocupaçoes.cvs')

In [ ]:
deputados_presidentes_legislatura_57 = superplanilha_com_cargos[(superplanilha_com_cargos['cargo'] == 'Presidente') & (superplanilha_com_cargos['Legislatura'] == 57)]

nomes_deputados_presidentes_legislatura_57 = deputados_presidentes_legislatura_57['Nome'].drop_duplicates().tolist()
nomes_deputados_presidentes_legislatura_57

In [ ]:
deputados_presidentes_legislatura_56 = superplanilha_com_cargos[(superplanilha_com_cargos['cargo'] == 'Presidente') & (superplanilha_com_cargos['Legislatura'] == 56)]

nomes_deputados_presidentes_legislatura_56= deputados_presidentes_legislatura_56['Nome'].drop_duplicates().tolist()
nomes_deputados_presidentes_legislatura_56

In [ ]:
deputados_presidentes_legislatura_55 = superplanilha_com_cargos[(superplanilha_com_cargos['cargo'] == 'Presidente') & (superplanilha_com_cargos['Legislatura'] == 55)]

nomes_deputados_presidentes_legislatura_55= deputados_presidentes_legislatura_55['Nome'].drop_duplicates().tolist()
nomes_deputados_presidentes_legislatura_55

<a id='secao2'></a>
## 2) Montando as Matrizes de correlações

In [ ]:
#função para criar matrizes :
def calcular_matriz_correlacao_deputados_por_legislatura(dataframe, legislatura):
    # Filtrar o DataFrame para a legislatura desejada
    legislatura_df = dataframe[dataframe['Legislatura'] == legislatura]
    
    # Pivote os dados para obter uma matriz de votos por deputado para cada sessão de votação
    votos_pivot = legislatura_df.pivot_table(index='idVotacao', columns='Nome', values='voto', aggfunc=lambda x: (x == 'Sim').sum() - (x == 'Não').sum())
    
 
    
    
    # Calcule a matriz de correlação para os votos dos deputados na legislatura atual
    correlation_matrix = votos_pivot.corr(method='spearman')
    
    
    return correlation_matrix

In [ ]:
#GURDAS AS TABELAS :
corre_57=calcular_matriz_correlacao_deputados_por_legislatura(superplanilha_com_cargos, 57)
corre_56=calcular_matriz_correlacao_deputados_por_legislatura(superplanilha_com_cargos, 56)
corre_55=calcular_matriz_correlacao_deputados_por_legislatura(superplanilha_com_cargos, 55)
corre_54=calcular_matriz_correlacao_deputados_por_legislatura(superplanilha_com_cargos, 54)
corre_53=calcular_matriz_correlacao_deputados_por_legislatura(superplanilha_com_cargos, 53)
corre_52=calcular_matriz_correlacao_deputados_por_legislatura(superplanilha_com_cargos, 52)

In [ ]:
corre_57=calcular_matriz_correlacao_deputados_por_legislatura(superplanilha_com_cargos, 57)
corre_57.head(50)

<a id='secao3'></a>
## 3) Lista dos 100 deputados com maiores nós.


In [ ]:
#deputado_correlation_matrix.head(1)

In [ ]:
#função para calculare os 100 maiores nós por legislatura
def calcular_top_deputados(dataframe, legislatura):
    # Filtrar o DataFrame para a legislatura desejada
    legislatura_df = dataframe[dataframe['Legislatura'] == legislatura]
    
    # Converter a coluna 'dataHoraVoto' para datetime e extrair o ano
    legislatura_df['Ano'] = pd.to_datetime(legislatura_df['dataHoraVoto']).dt.year
    
    # Pivote os dados para obter uma matriz de votos por deputado para cada sessão de votação
    votos_pivot = legislatura_df.pivot_table(index='idVotacao', columns='Nome', values='voto', 
                                             aggfunc=lambda x: (x == 'Sim').sum() - (x == 'Não').sum())
    
    # Calcule a matriz de correlação para os votos dos deputados na legislatura atual
    correlation_matrix = votos_pivot.corr()
    
    # Calcular a soma dos graus de correlação para cada deputado
    soma_graus = correlation_matrix.sum().sort_values(ascending=False)
    
    # Criar um DataFrame para a soma dos graus
    soma_graus_df = pd.DataFrame(soma_graus, columns=['Soma'])
    
    # Resetar o índice para obter o nome dos deputados como uma coluna
    soma_graus_df.reset_index(inplace=True)
    
    # Renomear a coluna para 'Nome'
    soma_graus_df.rename(columns={'index': 'Nome'}, inplace=True)
    
    # Mesclar com o DataFrame original para obter os cargos dos deputados e o ano
    top_deputados_df = pd.merge(soma_graus_df, legislatura_df[['Nome', 'cargo', 'Ano']], on='Nome', how='inner')
    
    # Remover duplicatas (caso haja algum deputado com mais de um cargo ou mais de um ano)
    top_deputados_df.drop_duplicates(inplace=True)
    
    # Agrupar por 'Nome' e calcular a média da 'Soma' para cada deputado
    top_deputados_df = top_deputados_df.groupby('Nome').agg({'Soma': 'mean', 'cargo': 'first', 'Ano': 'first'}).reset_index()
    
    # Ordenar pela soma dos graus em ordem decrescente e pegar os top 15 deputados
    top_deputados_df = top_deputados_df.sort_values(by='Soma', ascending=False).head(100)
    
    return top_deputados_df


In [ ]:
# gaurdar as variavies
top_100_deputados_57 = calcular_top_deputados(superplanilha_com_cargos, 57)
top_100_deputados_56 = calcular_top_deputados(superplanilha_com_cargos, 56)
top_100_deputados_55 = calcular_top_deputados(superplanilha_com_cargos, 55)
top_100_deputados_54 = calcular_top_deputados(superplanilha_com_cargos, 54)
top_100_deputados_53 = calcular_top_deputados(superplanilha_com_cargos, 53)
top_100_deputados_52 = calcular_top_deputados(superplanilha_com_cargos, 52)



In [ ]:
top_100_deputados_57.head(3) 
# Salvar as primeiras 30 linhas do DataFrame em um arquivo CSV


In [ ]:
top_100_deputados_57.head(30).to_csv(TABLE_DIR / 'top_30_57.csv', index=False)


In [ ]:
top_100_deputados_56.head(3) 


In [ ]:
top_100_deputados_56.head(30).to_csv(TABLE_DIR / 'top_30_56.csv', index=False)

In [ ]:
top_100_deputados_55.head(5)


In [ ]:
top_100_deputados_55.head(30).to_csv(TABLE_DIR / 'top_30_55.csv', index=False)

In [ ]:
top_100_deputados_54.head(5)

In [ ]:
top_100_deputados_54.head(30).to_csv(TABLE_DIR / 'top_30_54.csv', index=False)

In [ ]:
top_100_deputados_53.head(5)

In [ ]:
top_100_deputados_53.head(30).to_csv(TABLE_DIR / 'top_30_53.csv', index=False)

In [ ]:
top_100_deputados_52.head(5)

In [ ]:
top_100_deputados_52.head(30).to_csv(TABLE_DIR / 'top_30_52.csv', index=False)

In [ ]:
'''
# Lista de dataframes
dataframes = [top_100_deputados_52, top_100_deputados_53, top_100_deputados_54, top_100_deputados_55, top_100_deputados_56, top_100_deputados_57]

# Obter os nomes dos deputados em cada dataframe
nomes = [df['Nome'].tolist() for df in dataframes]

# Calcular a interseção dos nomes dos deputados
intersecao_nomes = set(nomes[0]).intersection(*nomes)

# Inicializar o dataframe final
df_final = pd.DataFrame(columns=['Nome', 'Quantidade', 'Anos'])

# Para cada nome na interseção, contar a quantidade de vezes que aparece e os anos correspondentes
for nome in intersecao_nomes:
    quantidade = 0
    anos = []
    for i, df in enumerate(dataframes):
        if nome in df['Nome'].tolist():
            quantidade += 1
            anos.append(52 + i)  # Supondo que o ano correspondente ao dataframe está incrementando de 1 em 1
    df_final = df_final.append({'Nome': nome, 'Quantidade': quantidade, 'Anos': anos}, ignore_index=True)

# Ordenar o dataframe pela quantidade em ordem decrescente
df_final = df_final.sort_values(by='Quantidade', ascending=False) '''


In [ ]:
arquivo_excel = DATA_DIR / 'Rankings Influência Deputados.xlsx'

# Crie um dicionário para armazenar os DataFrames, onde as chaves são os nomes das planilhas.
planilhas = {
    'Ranking 57': pd.read_excel(arquivo_excel, sheet_name='Ranking 57'),
    'Ranking 56': pd.read_excel(arquivo_excel, sheet_name='Ranking 56'),
    'Ranking 55': pd.read_excel(arquivo_excel, sheet_name='Ranking 55'),
    'Ranking 54': pd.read_excel(arquivo_excel, sheet_name='Ranking 54'),
    'Ranking 53': pd.read_excel(arquivo_excel, sheet_name='Ranking 53'),
    'Ranking 52': pd.read_excel(arquivo_excel, sheet_name='Ranking 52')
}

In [ ]:
ranking_57_df = planilhas['Ranking 57']
ranking_56_df = planilhas['Ranking 56']
ranking_55_df = planilhas['Ranking 55']
ranking_54_df = planilhas['Ranking 54']
ranking_53_df = planilhas['Ranking 53']
ranking_52_df = planilhas['Ranking 52']

In [ ]:
ranking_56_df = planilhas['Ranking 56']

In [ ]:
ranking_56_df.head(5)

In [ ]:
ranking_57_df.head(3) 

In [ ]:
# Lista das colunas a serem removidas
colunas_para_remover = ['Unnamed: 4', 'Unnamed: 5', 'Nome.1', 'Grau.1', 'Taxa de assiduidade.1', 'Grau ajustado.1']

ranking_57_df.drop(columns=colunas_para_remover,axis=0)


In [ ]:
ranking_57_df.head(3)

In [ ]:
ranking_56_df.head(3)

<a id='secao4'></a>
## 4) BoxPlot


### Boxplot 

In [ ]:
def create_normalized_boxplot(df, value_column, names_to_highlight):
    # Normalizando os valores para 513
    normalization_factor = 513 / len(df)
    df['Normalized Value'] = df[value_column] * normalization_factor
    
    # Convertendo valores para a escala de porcentagem
    min_grade = df['Normalized Value'].min()
    max_grade = df['Normalized Value'].max()
    df['Percentage Scale'] = ((df['Normalized Value'] - min_grade) / (max_grade - min_grade)) * 100

    # Criando o boxplot com a escala de porcentagem
    plt.figure(figsize=(6, 10))
    sns.boxplot(y='Percentage Scale', data=df, color='blue', width=0.6)

    # Marcar os quantis
    quantiles = df['Percentage Scale'].quantile([0.25, 0.5, 0.75, 0.9])
    colors = ['red', 'orange', 'yellow', 'green']
    quantile_labels = ['p25', 'p50', 'p75', 'p90']
    for quantile, color, label in zip(quantiles, colors, quantile_labels):
        plt.axhline(y=quantile, color=color, linestyle='--', label=label)  # Linhas tracejadas para os quantis

    # Destacar os nomes especificados com cores diferentes
    name_colors = sns.color_palette('husl', len(names_to_highlight))
    for name, color in zip(names_to_highlight, name_colors):
        value = df[df['Nome'] == name]['Percentage Scale'].values
        if value.size > 0:
            plt.plot(0, value[0], 'o', color=color, label=name, markersize=10)  # Adiciona um ponto colorido para o nome

    # Configurações adicionais
    plt.ylabel('Adjusted Grade (%)')
    plt.ylim(0, 100)  # Limitar o eixo y de 0% a 100%

    # Criar legenda com a mesma ordem das cores utilizadas
    handles, labels = plt.gca().get_legend_handles_labels()
    # Organize handles/labels so they match the order in the original list names_to_highlight
    ordered_handles = [handles[labels.index(name)] for name in names_to_highlight]
    ordered_labels = [label for label in names_to_highlight]
    # Add quantile handles/labels
    ordered_handles.extend(handles[:4])
    ordered_labels.extend(quantile_labels)
    plt.legend(handles=ordered_handles, labels=ordered_labels, title="")

    # Mostrar o plot
    plt.savefig(FIGURE_DIR / 'Boxplot.png',format='png', transparent=True)
    plt.tight_layout()
    plt.show()

In [ ]:
# Exemplo de uso da função
create_normalized_boxplot(ranking_57_df, 'Grau ajustado', ['Arthur Lira'])
#15Salvar a imagem como um arquivo PNG com fundo transparente


In [ ]:
# Exemplo de uso da função
create_normalized_boxplot(ranking_56_df, 'Grau ajustado', ['Arthur Lira', 'Rodrigo Maia'])


In [ ]:
create_normalized_boxplot(ranking_55_df, 'Grau Ajustado', ['Eduardo Cunha', 'Rodrigo Maia'])

In [ ]:
deputados_presidentes_legislatura_54 = superplanilha_com_cargos[(superplanilha_com_cargos['cargo'] == 'Presidente') & (superplanilha_com_cargos['Legislatura'] == 54)]

nomes_deputados_presidentes_legislatura_54= deputados_presidentes_legislatura_54['Nome'].drop_duplicates().tolist()
nomes_deputados_presidentes_legislatura_54

In [ ]:
create_normalized_boxplot(ranking_54_df, 'Grau ajustado', ['Henrique Eduardo Alves', 'Marco Maia'])

In [ ]:
deputados_presidentes_legislatura_53 = superplanilha_com_cargos[(superplanilha_com_cargos['cargo'] == 'Presidente') & (superplanilha_com_cargos['Legislatura'] == 53)]

nomes_deputados_presidentes_legislatura_53= deputados_presidentes_legislatura_53['Nome'].drop_duplicates().tolist()
nomes_deputados_presidentes_legislatura_53

In [ ]:
create_normalized_boxplot(ranking_53_df, 'Grau ajustado', ['Arlindo Chinaglia', 'Marco Maia', 'Michel Temer'])

In [ ]:
deputados_presidentes_legislatura_52 = superplanilha_com_cargos[(superplanilha_com_cargos['cargo'] == 'Presidente') & (superplanilha_com_cargos['Legislatura'] == 52)]

nomes_deputados_presidentes_legislatura_52= deputados_presidentes_legislatura_52['Nome'].drop_duplicates().tolist()
nomes_deputados_presidentes_legislatura_52

In [ ]:
create_normalized_boxplot(ranking_52_df, 'Grau ajustado', ['Aldo Rebelo', 'Arlindo Chinaglia', 'João Paulo Cunha', 'Severino Cavalcanti'])

In [ ]:
superplanilha_com_cargos.head(3)


In [ ]:
ranking_57_df.head(3)

In [ ]:
legislatura_57 = superplanilha_com_cargos[superplanilha_com_cargos['Legislatura'] == 57]
orientacao_esquerda = legislatura_57[legislatura_57['orientação'] == 'Esquerda']
df_merged = pd.merge(orientacao_esquerda, ranking_57_df, on='Nome')

# Suponha que df_merged é seu DataFrame que contém os dados combinados dos deputados com orientação 'Esquerda' na legislatura 57.

# Convertendo valores para a escala de porcentagem
min_grade = df_merged['Normalized Value'].min()
max_grade = df_merged['Normalized Value'].max()
df_merged['Percentage Scale'] = ((df_merged['Normalized Value'] - min_grade) / (max_grade - min_grade)) * 100

# Criando o boxplot com a escala de porcentagem
plt.figure(figsize=(6, 10))
sns.boxplot(y='Percentage Scale', data=df_merged, color='blue', width=0.6)

# Marcar os quantis
quantiles = df_merged['Percentage Scale'].quantile([0.25, 0.5, 0.75, 0.9])
colors = ['red', 'orange', 'yellow', 'green']
quantile_labels = ['25%', '50%', '75%', '90%']
for quantile, color, label in zip(quantiles, colors, quantile_labels):
    plt.axhline(y=quantile, color=color, linestyle='--', label=f'Quartil {label}')  # Linhas tracejadas para os quantis

# Configurações adicionais
plt.ylabel('Adjusted Grade (%)')
plt.ylim(0, 100)  # Limitar o eixo y de 0% a 100%

# Criar legenda com a mesma ordem das cores utilizadas
handles, labels = plt.gca().get_legend_handles_labels()
ordered_handles = [h for h, l in zip(handles, labels) if 'Quartil' in l]
ordered_labels = [l for l in labels if 'Quartil' in l]
plt.legend(handles=ordered_handles, labels=ordered_labels, title="Quartis", loc='upper right')

# Mostrar o plot
plt.tight_layout()
plt.show()


In [ ]:
legislatura_57 = superplanilha_com_cargos[superplanilha_com_cargos['Legislatura'] == 57]
orientacao_esquerda = legislatura_57[legislatura_57['orientação'] == 'Direita']
df_merged = pd.merge(orientacao_esquerda, ranking_57_df, on='Nome')

# Suponha que df_merged é seu DataFrame que contém os dados combinados dos deputados com orientação 'Esquerda' na legislatura 57.

# Convertendo valores para a escala de porcentagem
min_grade = df_merged['Normalized Value'].min()
max_grade = df_merged['Normalized Value'].max()
df_merged['Percentage Scale'] = ((df_merged['Normalized Value'] - min_grade) / (max_grade - min_grade)) * 100

# Criando o boxplot com a escala de porcentagem
plt.figure(figsize=(6, 10))
sns.boxplot(y='Percentage Scale', data=df_merged, color='blue', width=0.6)

# Marcar os quantis
quantiles = df_merged['Percentage Scale'].quantile([0.25, 0.5, 0.75, 0.9])
colors = ['red', 'orange', 'yellow', 'green']
quantile_labels = ['25%', '50%', '75%', '90%']
for quantile, color, label in zip(quantiles, colors, quantile_labels):
    plt.axhline(y=quantile, color=color, linestyle='--', label=f'Quartil {label}')  # Linhas tracejadas para os quantis

# Configurações adicionais
plt.ylabel('Adjusted Grade (%)')
plt.ylim(0, 100)  # Limitar o eixo y de 0% a 100%

# Criar legenda com a mesma ordem das cores utilizadas
handles, labels = plt.gca().get_legend_handles_labels()
ordered_handles = [h for h, l in zip(handles, labels) if 'Quartil' in l]
ordered_labels = [l for l in labels if 'Quartil' in l]
plt.legend(handles=ordered_handles, labels=ordered_labels, title="Quartis", loc='upper right')

# Mostrar o plot
plt.tight_layout()
plt.show()


In [ ]:
legislatura_57 = superplanilha_com_cargos[superplanilha_com_cargos['Legislatura'] == 57]
orientacao_esquerda = legislatura_57[legislatura_57['orientação'] == 'Centro-Direita']
df_merged = pd.merge(orientacao_esquerda, ranking_57_df, on='Nome')

# Suponha que df_merged é seu DataFrame que contém os dados combinados dos deputados com orientação 'Esquerda' na legislatura 57.

# Convertendo valores para a escala de porcentagem
min_grade = df_merged['Normalized Value'].min()
max_grade = df_merged['Normalized Value'].max()
df_merged['Percentage Scale'] = ((df_merged['Normalized Value'] - min_grade) / (max_grade - min_grade)) * 100

# Criando o boxplot com a escala de porcentagem
plt.figure(figsize=(6, 10))
sns.boxplot(y='Percentage Scale', data=df_merged, color='blue', width=0.6)

# Marcar os quantis
quantiles = df_merged['Percentage Scale'].quantile([0.25, 0.5, 0.75, 0.9])
colors = ['red', 'orange', 'yellow', 'green']
quantile_labels = ['25%', '50%', '75%', '90%']
for quantile, color, label in zip(quantiles, colors, quantile_labels):
    plt.axhline(y=quantile, color=color, linestyle='--', label=f'Quartil {label}')  # Linhas tracejadas para os quantis

# Configurações adicionais
plt.ylabel('Adjusted Grade (%)')
plt.ylim(0, 100)  # Limitar o eixo y de 0% a 100%

# Criar legenda com a mesma ordem das cores utilizadas
handles, labels = plt.gca().get_legend_handles_labels()
ordered_handles = [h for h, l in zip(handles, labels) if 'Quartil' in l]
ordered_labels = [l for l in labels if 'Quartil' in l]
plt.legend(handles=ordered_handles, labels=ordered_labels, title="Quartis", loc='upper right')

# Mostrar o plot
plt.tight_layout()
plt.show()

In [ ]:
legislatura_57 = superplanilha_com_cargos[superplanilha_com_cargos['Legislatura'] == 57]
orientacao_esquerda = legislatura_57[legislatura_57['orientação'] == 'Esquerda Moderada']
df_merged = pd.merge(orientacao_esquerda, ranking_57_df, on='Nome')

# Suponha que df_merged é seu DataFrame que contém os dados combinados dos deputados com orientação 'Esquerda' na legislatura 57.

# Convertendo valores para a escala de porcentagem
min_grade = df_merged['Normalized Value'].min()
max_grade = df_merged['Normalized Value'].max()
df_merged['Percentage Scale'] = ((df_merged['Normalized Value'] - min_grade) / (max_grade - min_grade)) * 100

# Criando o boxplot com a escala de porcentagem
plt.figure(figsize=(6, 10))
sns.boxplot(y='Percentage Scale', data=df_merged, color='blue', width=0.6)

# Marcar os quantis
quantiles = df_merged['Percentage Scale'].quantile([0.25, 0.5, 0.75, 0.9])
colors = ['red', 'orange', 'yellow', 'green']
quantile_labels = ['25%', '50%', '75%', '90%']
for quantile, color, label in zip(quantiles, colors, quantile_labels):
    plt.axhline(y=quantile, color=color, linestyle='--', label=f'Quartil {label}')  # Linhas tracejadas para os quantis

# Configurações adicionais
plt.ylabel('Adjusted Grade (%)')
plt.ylim(0, 100)  # Limitar o eixo y de 0% a 100%

# Criar legenda com a mesma ordem das cores utilizadas
handles, labels = plt.gca().get_legend_handles_labels()
ordered_handles = [h for h, l in zip(handles, labels) if 'Quartil' in l]
ordered_labels = [l for l in labels if 'Quartil' in l]
plt.legend(handles=ordered_handles, labels=ordered_labels, title="Quartis", loc='upper right')

# Mostrar o plot
plt.tight_layout()
plt.show()

<a id='secao5'></a>
## 5) centros de cada orientação

In [ ]:
# isso aqui a gabi já corrigiu 

In [ ]:
def imprimir_maior_deputado_por_orientacao_e_partido(superplanilha_com_cargos, corre_56):
    # Criar o grafo
    G = nx.Graph()

    # Adicionando os nós
    for deputado in corre_56.index:
        # Supõe-se que 'Nome', 'Partidos Ajustados', 'orientação' e 'cargo' são colunas do dataframe 'superplanilha_com_cargos'
        info_deputado = superplanilha_com_cargos[superplanilha_com_cargos['Nome'] == deputado].iloc[0]
        G.add_node(deputado, 
                   partido=info_deputado['Partidos Ajustados'],
                   orientacao=info_deputado['orientação'],
                   cargo=info_deputado['cargo'])

    # Adicionando as arestas com base na correlação
    for i, row in corre_56.iterrows():
        for j, value in row.iteritems():
            if i != j and value > 0.7:  # supondo que 0.7 seja o threshold de correlação significativa
                G.add_edge(i, j, weight=value)

    # Calcular o grau de cada nó
    degree = dict(G.degree(weight='weight'))

    # Encontrar o maior nó de cada orientação
    orientacoes = superplanilha_com_cargos['orientação'].unique()
    maior_deputado_por_orientacao = {}
    for orientacao in orientacoes:
        maior_deputado = None
        maior_grau = -1
        for no, dados in G.nodes(data=True):
            if dados['orientacao'] == orientacao and degree[no] > maior_grau:
                maior_deputado = no
                maior_grau = degree[no]
        if maior_deputado:
            maior_deputado_por_orientacao[orientacao] = (maior_deputado, maior_grau)

    # Imprimir o maior deputado de cada orientação
    for orientacao, (deputado, grau) in maior_deputado_por_orientacao.items():
        partido = G.nodes[deputado]['partido']
        print(f"Maior deputado da orientação {orientacao}: {deputado}, Partido: {partido}, Grau: {grau}")

    # Encontrar o maior deputado do partido MDB
    maior_deputado_mdb = None
    maior_grau_mdb = -1
    for no, dados in G.nodes(data=True):
        if dados['partido'] == 'MDB' and degree[no] > maior_grau_mdb:
            maior_deputado_mdb = no
            maior_grau_mdb = degree[no]

    # Imprimir o maior deputado do MDB
    if maior_deputado_mdb:
        print(f"Maior deputado do MDB: {maior_deputado_mdb}, Grau: {maior_grau_mdb}")

# Exemplo de como chamar a função




In [ ]:
# Exemplo de como chamar a função
imprimir_maior_deputado_por_orientacao_e_partido(superplanilha_com_cargos, corre_57)

In [ ]:
imprimir_maior_deputado_por_orientacao_e_partido(superplanilha_com_cargos, corre_56)

In [ ]:
imprimir_maior_deputado_por_orientacao_e_partido(superplanilha_com_cargos, corre_55)

In [ ]:
imprimir_maior_deputado_por_orientacao_e_partido(superplanilha_com_cargos, corre_54)

In [ ]:
imprimir_maior_deputado_por_orientacao_e_partido(superplanilha_com_cargos, corre_53)

In [ ]:
imprimir_maior_deputado_por_orientacao_e_partido(superplanilha_com_cargos, corre_52)

### distribuição das correlações

In [ ]:
def plot_histograma_correlacao_legislatura(dataframe, legislatura, bins=100):
    # Primeiro, calcular a matriz de correlação
    matriz_correlacao = calcular_matriz_correlacao_deputados_por_legislatura(dataframe, legislatura)
    
    # Criar a máscara para o triângulo superior
    mask = np.triu(np.ones_like(matriz_correlacao, dtype=bool), k=1)
    
    # Aplicar a máscara para extrair os valores de correlação
    valores_correlacao = matriz_correlacao[mask]
    
    # Remover NaNs
    valores_correlacao = valores_correlacao[~np.isnan(valores_correlacao)]
    
    # Criar o histograma
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.hist(valores_correlacao, bins=bins)
    ax.set_title(f"Distribuição das correlações: apenas acordos para a {legislatura}ª legislatura")
    ax.set_xlabel('Correlação')
    ax.set_ylabel('Frequência')
    
    # Mostrar o histograma
    plt.show()
    # Retorna o objeto da figura se necessário
    return fig




In [ ]:

# Presumindo que a função calcular_matriz_correlacao_deputados_por_legislatura já esteja definida

def plot_correlation_distribution(df, legislature_number):
    # Calcula a matriz de correlação para a legislatura especificada
    correlation_matrix = calcular_matriz_correlacao_deputados_por_legislatura(df, legislature_number)
    
    # Cria uma máscara para ignorar a diagonal principal
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool), k=1)
    
    # Seleciona os valores acima da diagonal principal (sem incluir a diagonal)
    correlation_values = correlation_matrix.where(mask).stack().dropna()

    # Cria o histograma
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.hist(correlation_values, bins=100)
    ax.set_title(f"Distribuição das correlações: apenas acordos para a {legislature_number}ª legislatura")
    ax.set_xlabel('Correlação')
    ax.set_ylabel('Frequência')
    plt.show()

    return fig, ax  # Retorna o objeto da figura e dos eixos para uso adicional se necessário




In [ ]:
def calcular_grau_deputados_matriz(matriz_correlacao):
    # Criar o grafo
    G = nx.Graph()

    # Adicionando os nós com base nos nomes dos deputados na matriz de correlação
    for nome in matriz_correlacao.index:
        G.add_node(nome)

    # Adicionando as arestas com base na matriz de correlação
    for deputado1 in matriz_correlacao.index:
        for deputado2 in matriz_correlacao.columns:
            # Adicionar todas as correlações positivas
            if deputado1 != deputado2:
                peso = matriz_correlacao.at[deputado1, deputado2]
                if peso > 0:
                    G.add_edge(deputado1, deputado2, weight=peso)

    # Calcular o grau ponderado de cada nó
    graus = {node: sum(data['weight'] for _, _, data in G.edges(node, data=True)) for node in G}

    # Criar DataFrame a partir dos graus
    df_graus = pd.DataFrame.from_dict(graus, orient='index', columns=['Grau']).reset_index()
    df_graus.rename(columns={'index': 'Nome'}, inplace=True)

    # Ordenar o DataFrame pelo grau em ordem decrescente
    df_graus_ordenado = df_graus.sort_values('Grau', ascending=False)

    return df_graus_ordenado

# Exemplo de uso da função:
# corre_57 = ... # Sua matriz de correlação com 535 nomes
# df_graus_deputados = calcular_grau_deputados_matriz(corre_57)
# print(df_graus_deputados)


In [ ]:
df_57= calcular_grau_deputados_matriz(corre_57)
df_56= calcular_grau_deputados_matriz(corre_56)
df_55= calcular_grau_deputados_matriz(corre_55)
df_54= calcular_grau_deputados_matriz(corre_54)
df_53= calcular_grau_deputados_matriz(corre_53)
df_52= calcular_grau_deputados_matriz(corre_52)